In [1]:
# ===========================
# CELL 1: Install and Import Required Libraries
# ===========================

# Install NLTK if not already available (run in Google Colab)
!pip install nltk

# Import the three core libraries we'll use throughout
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re
import math

# Download necessary NLTK data - these are essential datasets for text processing
nltk.download('punkt')          # For tokenization (splitting text into words/sentences)
nltk.download('stopwords')      # For common words like 'the', 'and', 'is'
nltk.download('averaged_perceptron_tagger')  # For part-of-speech tagging
nltk.download('wordnet')        # For lemmatization (finding root word forms)
nltk.download('vader_lexicon')  # For sentiment analysis
nltk.download('omw-1.4')       # Additional wordnet data


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [2]:
# ===========================
# CELL 2: Creating Our Text Dataset
# ===========================

print("=== BUILDING OUR TEXT DATASET ===")
print("Let's create a diverse collection of texts to work with throughout our examples")
print("This will help us see how different NLP techniques work on various types of content\n")

# Create a comprehensive dataset with different types of text
# This variety helps us understand how NLP techniques work across different domains
sample_texts = [
    "I absolutely love this new smartphone! The camera quality is amazing and the battery lasts all day. Best purchase I've made this year.",
    "The movie was terrible. Poor acting, weak plot, and the ending made no sense. I want my money back.",
    "Apple Inc. announced its quarterly earnings today. The company reported revenue of $89.5 billion for Q3 2023 in Cupertino, California.",
    "The weather is beautiful today in New York City. I'm planning to visit Central Park and maybe grab lunch at a nice restaurant.",
    "Machine learning and artificial intelligence are transforming the way we work. Python is becoming the most popular programming language for data science.",
    "The new restaurant downtown serves excellent Italian food. The pasta was perfectly cooked and the service was outstanding.",
    "Climate change is a serious global issue. Rising temperatures are affecting weather patterns worldwide, causing severe droughts and floods.",
    "I can't believe how fast technology is advancing. Virtual reality and augmented reality are becoming mainstream in gaming and education.",
    "The book I'm reading is fascinating. It's about the history of computer science and early programming languages like FORTRAN and COBOL.",
    "My laptop is running slowly today. I think I need to update the software and clean up some files to improve performance."
]

# Create structured data for more complex analysis
# This simulates real-world datasets you might encounter
news_data = {
    'text': [
        "President Biden met with European leaders in Brussels yesterday to discuss economic cooperation and security measures.",
        "Tesla announced plans to build a new manufacturing facility in Texas, creating thousands of jobs for local workers.",
        "Scientists at MIT have developed a new battery technology that could revolutionize electric vehicle performance and range.",
        "The Federal Reserve decided to raise interest rates by 0.25% to combat rising inflation in the economy.",
        "Google's new AI chatbot demonstrates impressive capabilities in natural language understanding and generation tasks.",
        "Amazon reported strong quarterly earnings driven by cloud computing services and online retail growth.",
        "Researchers have discovered a new species of deep-sea fish in the Pacific Ocean near the Mariana Trench.",
        "The Supreme Court heard arguments today regarding digital privacy rights and government surveillance powers."
    ],
    'category': ['Politics', 'Business', 'Technology', 'Economics', 'Technology', 'Business', 'Science', 'Politics'],
    'sentiment': ['Neutral', 'Positive', 'Positive', 'Neutral', 'Positive', 'Positive', 'Neutral', 'Neutral']
}

# Convert to pandas DataFrame for easier manipulation
df_news = pd.DataFrame(news_data)

print("Sample texts created! Here's what we're working with:")
print(f"Number of sample texts: {len(sample_texts)}")
print(f"News dataset shape: {df_news.shape}")
print("\nFirst three sample texts:")
for i, text in enumerate(sample_texts[:3]):
    print(f"{i+1}. {text}")

print(f"\nNews DataFrame preview:")
print(df_news.head())


=== BUILDING OUR TEXT DATASET ===
Let's create a diverse collection of texts to work with throughout our examples
This will help us see how different NLP techniques work on various types of content

Sample texts created! Here's what we're working with:
Number of sample texts: 10
News dataset shape: (8, 3)

First three sample texts:
1. I absolutely love this new smartphone! The camera quality is amazing and the battery lasts all day. Best purchase I've made this year.
2. The movie was terrible. Poor acting, weak plot, and the ending made no sense. I want my money back.
3. Apple Inc. announced its quarterly earnings today. The company reported revenue of $89.5 billion for Q3 2023 in Cupertino, California.

News DataFrame preview:
                                                text    category sentiment
0  President Biden met with European leaders in B...    Politics   Neutral
1  Tesla announced plans to build a new manufactu...    Business  Positive
2  Scientists at MIT have developed a

In [3]:
# Let's analyze the basic characteristics of our dataset
total_chars = sum(len(text) for text in sample_texts)
avg_chars = total_chars / len(sample_texts)
print(f"\nDataset statistics:")
print(f"Total characters: {total_chars}")
print(f"Average characters per text: {avg_chars:.1f}")



Dataset statistics:
Total characters: 1298
Average characters per text: 129.8


In [4]:
# ===========================
# CELL 3: Tokenization - Breaking Text into Pieces
# ===========================

print("=== TOKENIZATION: THE FOUNDATION OF TEXT PROCESSING ===")
print("Tokenization is like taking apart a watch to understand its components")
print("We break text into meaningful units that computers can work with\n")

from nltk.tokenize import word_tokenize, sent_tokenize

# Let's start with a complex example to show tokenization challenges
complex_text = "Dr. Smith, who works at U.S.A. Inc., said, \"I can't believe it's 2023! The A.I. revolution is here.\""
print(f"Complex example: {complex_text}")


=== TOKENIZATION: THE FOUNDATION OF TEXT PROCESSING ===
Tokenization is like taking apart a watch to understand its components
We break text into meaningful units that computers can work with

Complex example: Dr. Smith, who works at U.S.A. Inc., said, "I can't believe it's 2023! The A.I. revolution is here."


In [6]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
# Demonstrate different tokenization approaches
print("\n1. Simple split() tokenization:")
simple_tokens = complex_text.split()
print(f"Result: {simple_tokens}")
print(f"Issues: Notice how punctuation sticks to words, and contractions aren't handled well")

print("\n2. NLTK word tokenization (smarter approach):")
nltk_tokens = word_tokenize(complex_text)
print(f"Result: {nltk_tokens}")
print(f"Better: Punctuation is separated, contractions are handled properly")

print("\n3. NLTK sentence tokenization:")
sentences = sent_tokenize(complex_text)
print(f"Result: {sentences}")
print(f"Smart: Understands that 'Dr.' and 'U.S.A.' are not sentence endings")


1. Simple split() tokenization:
Result: ['Dr.', 'Smith,', 'who', 'works', 'at', 'U.S.A.', 'Inc.,', 'said,', '"I', "can't", 'believe', "it's", '2023!', 'The', 'A.I.', 'revolution', 'is', 'here."']
Issues: Notice how punctuation sticks to words, and contractions aren't handled well

2. NLTK word tokenization (smarter approach):
Result: ['Dr.', 'Smith', ',', 'who', 'works', 'at', 'U.S.A.', 'Inc.', ',', 'said', ',', '``', 'I', 'ca', "n't", 'believe', 'it', "'s", '2023', '!', 'The', 'A.I', '.', 'revolution', 'is', 'here', '.', "''"]
Better: Punctuation is separated, contractions are handled properly

3. NLTK sentence tokenization:
Result: ['Dr. Smith, who works at U.S.A. Inc., said, "I can\'t believe it\'s 2023!', 'The A.I.', 'revolution is here."']
Smart: Understands that 'Dr.' and 'U.S.A.' are not sentence endings


In [8]:
# Let's create a comprehensive tokenization function
def comprehensive_tokenize(text, remove_punctuation=True, lowercase=True):
    """
    A comprehensive tokenization function that handles various text preprocessing steps

    Parameters:
    text: Input text to tokenize
    remove_punctuation: Whether to filter out punctuation marks
    lowercase: Whether to convert all tokens to lowercase

    Returns:
    List of cleaned tokens
    """
    # Start with NLTK tokenization for proper handling of contractions and punctuation
    tokens = word_tokenize(text)

    # Apply lowercase conversion if requested
    if lowercase:
        tokens = [token.lower() for token in tokens]

    # Remove punctuation if requested - keep only alphabetic characters
    if remove_punctuation:
        tokens = [token for token in tokens if token.isalpha()]

    return tokens


In [9]:
# Apply our tokenization function to sample texts
print("\n=== APPLYING TOKENIZATION TO OUR DATASET ===")
all_tokens = []  # We'll collect all tokens for analysis

for i, text in enumerate(sample_texts[:3]):  # Just first 3 for demonstration
    tokens = comprehensive_tokenize(text)
    all_tokens.extend(tokens)

    print(f"\nText {i+1}: {text[:60]}...")
    print(f"Tokens ({len(tokens)}): {tokens[:10]}...")  # Show first 10 tokens

    # Calculate some basic statistics
    avg_word_length = np.mean([len(token) for token in tokens])
    print(f"Average word length: {avg_word_length:.1f} characters")



=== APPLYING TOKENIZATION TO OUR DATASET ===

Text 1: I absolutely love this new smartphone! The camera quality is...
Tokens (23): ['i', 'absolutely', 'love', 'this', 'new', 'smartphone', 'the', 'camera', 'quality', 'is']...
Average word length: 4.6 characters

Text 2: The movie was terrible. Poor acting, weak plot, and the endi...
Tokens (19): ['the', 'movie', 'was', 'terrible', 'poor', 'acting', 'weak', 'plot', 'and', 'the']...
Average word length: 4.0 characters

Text 3: Apple Inc. announced its quarterly earnings today. The compa...
Tokens (16): ['apple', 'announced', 'its', 'quarterly', 'earnings', 'today', 'the', 'company', 'reported', 'revenue']...
Average word length: 6.1 characters


In [10]:
# Overall tokenization statistics
print(f"\n=== TOKENIZATION STATISTICS ===")
print(f"Total tokens collected: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}")
print(f"Vocabulary richness: {len(set(all_tokens))/len(all_tokens):.3f}")



=== TOKENIZATION STATISTICS ===
Total tokens collected: 58
Unique tokens: 49
Vocabulary richness: 0.845


In [11]:
# Most common tokens
token_freq = Counter(all_tokens)
print(f"\nMost frequent tokens:")
for token, freq in token_freq.most_common(10):
    print(f"  {token}: {freq}")



Most frequent tokens:
  the: 5
  i: 3
  this: 2
  and: 2
  made: 2
  absolutely: 1
  love: 1
  new: 1
  smartphone: 1
  camera: 1


In [12]:
# ===========================
# CELL 4: Stop Words Removal - Filtering the Noise
# ===========================

print("=== STOP WORDS REMOVAL: FINDING THE SIGNAL IN THE NOISE ===")
print("Stop words are like background music - they're everywhere but don't carry the main meaning")
print("Removing them helps us focus on the content that really matters\n")

from nltk.corpus import stopwords

# Get the standard English stop words from NLTK
english_stopwords = set(stopwords.words('english'))
print(f"NLTK provides {len(english_stopwords)} English stop words")
print(f"Examples: {list(english_stopwords)[:15]}")

# Let's also create our own custom stop word list for demonstration
custom_stopwords = english_stopwords.union({'would', 'could', 'should', 'might', 'also', 'really'})
print(f"\nWith custom additions: {len(custom_stopwords)} stop words")


=== STOP WORDS REMOVAL: FINDING THE SIGNAL IN THE NOISE ===
Stop words are like background music - they're everywhere but don't carry the main meaning
Removing them helps us focus on the content that really matters

NLTK provides 198 English stop words
Examples: ["we've", 'am', 'and', 'other', 'no', "wouldn't", 's', 'his', "i'd", 'isn', 'too', "isn't", 'have', 'about', 'in']

With custom additions: 203 stop words


In [13]:
def remove_stopwords(tokens, stopwords_set=english_stopwords):
    """
    Remove stop words from a list of tokens

    Parameters:
    tokens: List of word tokens
    stopwords_set: Set of words to remove (default is NLTK English stop words)

    Returns:
    List of tokens with stop words removed
    """
    return [token for token in tokens if token.lower() not in stopwords_set]

# Demonstrate the effect of stop word removal
demo_text = sample_texts[0]  # Use first sample text
print(f"\n=== STOP WORD REMOVAL DEMONSTRATION ===")
print(f"Original text: {demo_text}")

# Tokenize the text
original_tokens = comprehensive_tokenize(demo_text)
print(f"\nOriginal tokens ({len(original_tokens)}): {original_tokens}")

# Remove stop words
filtered_tokens = remove_stopwords(original_tokens)
print(f"\nAfter removing stop words ({len(filtered_tokens)}): {filtered_tokens}")

# Show what was removed
removed_words = [token for token in original_tokens if token not in filtered_tokens]
print(f"\nRemoved words: {removed_words}")
print(f"Reduction: {len(removed_words)} words ({len(removed_words)/len(original_tokens)*100:.1f}%)")

# Apply stop word removal to all sample texts and analyze the impact
print(f"\n=== IMPACT ANALYSIS ACROSS ALL TEXTS ===")
total_original = 0
total_filtered = 0

for i, text in enumerate(sample_texts):
    tokens = comprehensive_tokenize(text)
    filtered = remove_stopwords(tokens)

    total_original += len(tokens)
    total_filtered += len(filtered)

    reduction = (len(tokens) - len(filtered)) / len(tokens) * 100
    print(f"Text {i+1}: {len(tokens)} -> {len(filtered)} tokens ({reduction:.1f}% reduction)")

overall_reduction = (total_original - total_filtered) / total_original * 100
print(f"\nOverall reduction: {overall_reduction:.1f}%")
print(f"This means we removed {overall_reduction:.1f}% of words while keeping the meaningful content!")

# Let's see how stop word removal affects word frequency
print(f"\n=== WORD FREQUENCY BEFORE AND AFTER STOP WORD REMOVAL ===")

# Collect all tokens from sample texts
all_original_tokens = []
all_filtered_tokens = []

for text in sample_texts:
    tokens = comprehensive_tokenize(text)
    all_original_tokens.extend(tokens)
    all_filtered_tokens.extend(remove_stopwords(tokens))

# Calculate frequencies
original_freq = Counter(all_original_tokens)
filtered_freq = Counter(all_filtered_tokens)

print("Top 10 words BEFORE stop word removal:")
for word, count in original_freq.most_common(10):
    print(f"  {word}: {count}")

print("\nTop 10 words AFTER stop word removal:")
for word, count in filtered_freq.most_common(10):
    print(f"  {word}: {count}")



=== STOP WORD REMOVAL DEMONSTRATION ===
Original text: I absolutely love this new smartphone! The camera quality is amazing and the battery lasts all day. Best purchase I've made this year.

Original tokens (23): ['i', 'absolutely', 'love', 'this', 'new', 'smartphone', 'the', 'camera', 'quality', 'is', 'amazing', 'and', 'the', 'battery', 'lasts', 'all', 'day', 'best', 'purchase', 'i', 'made', 'this', 'year']

After removing stop words (14): ['absolutely', 'love', 'new', 'smartphone', 'camera', 'quality', 'amazing', 'battery', 'lasts', 'day', 'best', 'purchase', 'made', 'year']

Removed words: ['i', 'this', 'the', 'is', 'and', 'the', 'all', 'i', 'this']
Reduction: 9 words (39.1%)

=== IMPACT ANALYSIS ACROSS ALL TEXTS ===
Text 1: 23 -> 14 tokens (39.1% reduction)
Text 2: 19 -> 12 tokens (36.8% reduction)
Text 3: 16 -> 11 tokens (31.2% reduction)
Text 4: 23 -> 15 tokens (34.8% reduction)
Text 5: 22 -> 14 tokens (36.4% reduction)
Text 6: 18 -> 12 tokens (33.3% reduction)
Text 7: 19 -> 15 

In [16]:
# ===========================
# CELL 5: Parts of Speech Tagging - Understanding Grammatical Roles
# ===========================

print("=== PARTS OF SPEECH TAGGING: UNDERSTANDING WORD ROLES ===")
print("POS tagging is like identifying the role each actor plays in a sentence")
print("It helps computers understand the grammatical structure of language\n")

from nltk import pos_tag

# Create a comprehensive POS tag dictionary for better understanding
pos_tag_meanings = {
    'CC': 'Coordinating conjunction (and, but, or)',
    'CD': 'Cardinal number (one, two, three)',
    'DT': 'Determiner (the, a, an)',
    'EX': 'Existential there',
    'FW': 'Foreign word',
    'IN': 'Preposition/subordinating conjunction (in, of, like)',
    'JJ': 'Adjective (big, blue, fast)',
    'JJR': 'Adjective, comparative (bigger, faster)',
    'JJS': 'Adjective, superlative (biggest, fastest)',
    'LS': 'List item marker',
    'MD': 'Modal (can, could, may, must)',
    'NN': 'Noun, singular (cat, tree, idea)',
    'NNS': 'Noun, plural (cats, trees, ideas)',
    'NNP': 'Proper noun, singular (John, London)',
    'NNPS': 'Proper noun, plural (Americans, Indians)',
    'PDT': 'Predeterminer (all, both, half)',
    'POS': 'Possessive ending (\'s)',
    'PRP': 'Personal pronoun (I, he, she, it)',
    'PRP$': 'Possessive pronoun (my, his, her)',
    'RB': 'Adverb (quickly, very, however)',
    'RBR': 'Adverb, comparative (faster, better)',
    'RBS': 'Adverb, superlative (fastest, best)',
    'RP': 'Particle (give up, put off)',
    'TO': 'to',
    'UH': 'Interjection (uh, wow, hello)',
    'VB': 'Verb, base form (eat, go, run)',
    'VBD': 'Verb, past tense (ate, went, ran)',
    'VBG': 'Verb, gerund/present participle (eating, going)',
    'VBN': 'Verb, past participle (eaten, gone)',
    'VBP': 'Verb, present tense, not 3rd person singular (eat, go)',
    'VBZ': 'Verb, present tense, 3rd person singular (eats, goes)',
    'WDT': 'Wh-determiner (which, that)',
    'WP': 'Wh-pronoun (who, what)',
    'WP$': 'Possessive wh-pronoun (whose)',
    'WRB': 'Wh-adverb (where, when, why, how)'
}

def get_pos_meaning(tag):
    """Get human-readable meaning for POS tag"""
    return pos_tag_meanings.get(tag, f"Unknown tag: {tag}")

# Demonstrate POS tagging with a complex sentence
example_sentence = "The brilliant scientist quickly discovered three amazing new compounds yesterday."
print(f"Example sentence: {example_sentence}")

# Tokenize and tag
tokens = word_tokenize(example_sentence)
pos_tags = pos_tag(tokens)

print(f"\nPOS Analysis:")
print(f"{'Word':<12} {'Tag':<6} {'Meaning'}")
print("-" * 50)
for word, tag in pos_tags:
    meaning = get_pos_meaning(tag)
    print(f"{word:<12} {tag:<6} {meaning}")

# Function to analyze POS distribution in text
def analyze_pos_distribution(text, title="Text"):
    """
    Analyze the distribution of parts of speech in a given text

    Parameters:
    text: Input text to analyze
    title: Title for the analysis output

    Returns:
    Dictionary with POS tag counts and percentages
    """
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)

    # Count POS tags
    pos_counts = Counter([tag for word, tag in pos_tags])
    total_words = len(tokens)

    print(f"\n=== POS ANALYSIS: {title} ===")
    print(f"Text preview: {text[:60]}...")
    print(f"Total words: {total_words}")

    # Group by major categories for easier understanding
    nouns = sum(count for tag, count in pos_counts.items() if tag.startswith('N'))
    verbs = sum(count for tag, count in pos_counts.items() if tag.startswith('V'))
    adjectives = sum(count for tag, count in pos_counts.items() if tag.startswith('J'))
    adverbs = sum(count for tag, count in pos_counts.items() if tag.startswith('R'))

    print(f"\nMajor categories:")
    print(f"  Nouns: {nouns} ({nouns/total_words*100:.1f}%)")
    print(f"  Verbs: {verbs} ({verbs/total_words*100:.1f}%)")
    print(f"  Adjectives: {adjectives} ({adjectives/total_words*100:.1f}%)")
    print(f"  Adverbs: {adverbs} ({adverbs/total_words*100:.1f}%)")

    print(f"\nMost common specific tags:")
    for tag, count in pos_counts.most_common(5):
        percentage = count / total_words * 100
        meaning = get_pos_meaning(tag)
        print(f"  {tag}: {count} ({percentage:.1f}%) - {meaning}")

    return pos_counts

# Analyze different types of texts to see how POS patterns vary
analyze_pos_distribution(sample_texts[0], "Product Review (Emotional)")
analyze_pos_distribution(sample_texts[2], "News Article (Factual)")
analyze_pos_distribution(sample_texts[4], "Technical Text (Descriptive)")

# Analyze POS patterns across our entire dataset
print(f"\n=== OVERALL POS PATTERNS IN DATASET ===")
all_pos_tags = []

for text in sample_texts:
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    all_pos_tags.extend([tag for word, tag in pos_tags])

overall_pos_counts = Counter(all_pos_tags)
total_tags = len(all_pos_tags)

print(f"Total words analyzed: {total_tags}")
print(f"Unique POS tags found: {len(overall_pos_counts)}")

# Show the most common patterns
print(f"\nMost common POS patterns in our dataset:")
for tag, count in overall_pos_counts.most_common(10):
    percentage = count / total_tags * 100
    meaning = get_pos_meaning(tag)
    print(f"  {tag}: {count} ({percentage:.1f}%) - {meaning}")

# Extract and display examples of different word types
def extract_word_types(texts, pos_filter):
    """Extract words of specific POS types from texts"""
    words = []
    for text in texts:
        tokens = word_tokenize(text)
        pos_tags = pos_tag(tokens)
        words.extend([word.lower() for word, tag in pos_tags if tag.startswith(pos_filter)])
    return list(set(words))  # Remove duplicates

print(f"\n=== EXAMPLES OF DIFFERENT WORD TYPES ===")
common_nouns = extract_word_types(sample_texts, 'NN')[:10]
common_verbs = extract_word_types(sample_texts, 'VB')[:10]
common_adjectives = extract_word_types(sample_texts, 'JJ')[:10]

print(f"Common nouns found: {common_nouns}")
print(f"Common verbs found: {common_verbs}")
print(f"Common adjectives found: {common_adjectives}")


=== PARTS OF SPEECH TAGGING: UNDERSTANDING WORD ROLES ===
POS tagging is like identifying the role each actor plays in a sentence
It helps computers understand the grammatical structure of language

Example sentence: The brilliant scientist quickly discovered three amazing new compounds yesterday.

POS Analysis:
Word         Tag    Meaning
--------------------------------------------------
The          DT     Determiner (the, a, an)
brilliant    JJ     Adjective (big, blue, fast)
scientist    NN     Noun, singular (cat, tree, idea)
quickly      RB     Adverb (quickly, very, however)
discovered   VBD    Verb, past tense (ate, went, ran)
three        CD     Cardinal number (one, two, three)
amazing      VBG    Verb, gerund/present participle (eating, going)
new          JJ     Adjective (big, blue, fast)
compounds    NNS    Noun, plural (cats, trees, ideas)
yesterday    NN     Noun, singular (cat, tree, idea)
.            .      Unknown tag: .

=== POS ANALYSIS: Product Review (Emotional

In [17]:
# ===========================
# CELL 6: Stemming - Finding Word Roots with Rules
# ===========================

print("=== STEMMING: FINDING WORD ROOTS THROUGH ALGORITHMIC RULES ===")
print("Stemming is like pruning a tree - we cut back to the main trunk")
print("It uses simple rules to remove prefixes and suffixes from words\n")

from nltk.stem import PorterStemmer, SnowballStemmer

# Initialize different stemming algorithms
porter_stemmer = PorterStemmer()
snowball_stemmer = SnowballStemmer('english')

# Comprehensive test words to demonstrate stemming behavior
test_word_groups = [
    # Regular verb forms
    ['run', 'running', 'runs', 'ran', 'runner'],
    # Study/education related
    ['study', 'studying', 'studies', 'studied', 'student'],
    # Connection/communication
    ['connect', 'connection', 'connected', 'connecting', 'connects'],
    # Beauty/appearance
    ['beauty', 'beautiful', 'beautifully', 'beautify'],
    # Work/business
    ['work', 'working', 'worked', 'worker', 'works'],
    # Happy/emotion
    ['happy', 'happiness', 'happily', 'happier', 'happiest']
]

print("=== COMPREHENSIVE STEMMING DEMONSTRATION ===")
print(f"{'Original Word':<15} {'Porter Stem':<12} {'Snowball Stem':<15} {'Notes'}")
print("-" * 70)

for word_group in test_word_groups:
    print(f"\nWord family: {word_group[0]}")
    for word in word_group:
        porter_result = porter_stemmer.stem(word)
        snowball_result = snowball_stemmer.stem(word)

        # Add notes about the stemming quality
        notes = ""
        if porter_result == snowball_result:
            notes = "Same result"
        elif len(porter_result) < len(snowball_result):
            notes = "Porter more aggressive"
        else:
            notes = "Snowball more aggressive"

        print(f"  {word:<13} {porter_result:<12} {snowball_result:<15} {notes}")

# Create a comprehensive stemming function
def stem_text(text, stemmer_type='porter'):
    """
    Stem all words in a text using specified stemmer

    Parameters:
    text: Input text to stem
    stemmer_type: 'porter' or 'snowball'

    Returns:
    List of stemmed tokens
    """
    # Choose stemmer
    if stemmer_type == 'porter':
        stemmer = porter_stemmer
    else:
        stemmer = snowball_stemmer

    # Tokenize and stem only alphabetic words
    tokens = word_tokenize(text.lower())
    stemmed_tokens = []

    for token in tokens:
        if token.isalpha():  # Only stem actual words, not punctuation
            stemmed_tokens.append(stemmer.stem(token))

    return stemmed_tokens

# Apply stemming to our sample texts
print(f"\n=== STEMMING APPLIED TO SAMPLE TEXTS ===")

for i, text in enumerate(sample_texts[:3]):  # First 3 texts for demonstration
    print(f"\nText {i+1}: {text[:50]}...")

    # Get original tokens
    original_tokens = comprehensive_tokenize(text)

    # Get stemmed versions
    porter_stemmed = stem_text(text, 'porter')
    snowball_stemmed = stem_text(text, 'snowball')

    print(f"Original  ({len(original_tokens):2d}): {' '.join(original_tokens[:8])}...")
    print(f"Porter    ({len(porter_stemmed):2d}): {' '.join(porter_stemmed[:8])}...")
    print(f"Snowball  ({len(snowball_stemmed):2d}): {' '.join(snowball_stemmed[:8])}...")

    # Calculate vocabulary reduction
    original_vocab = len(set(original_tokens))
    porter_vocab = len(set(porter_stemmed))
    snowball_vocab = len(set(snowball_stemmed))

    porter_reduction = (original_vocab - porter_vocab) / original_vocab * 100
    snowball_reduction = (original_vocab - snowball_vocab) / original_vocab * 100

    print(f"Vocabulary: {original_vocab} -> Porter: {porter_vocab} ({porter_reduction:.1f}% reduction)")
    print(f"                    -> Snowball: {snowball_vocab} ({snowball_reduction:.1f}% reduction)")

# Comprehensive vocabulary analysis across all texts
print(f"\n=== VOCABULARY REDUCTION ANALYSIS ===")

all_original_words = []
all_porter_stems = []
all_snowball_stems = []

for text in sample_texts:
    original_tokens = comprehensive_tokenize(text)
    all_original_words.extend(original_tokens)

    porter_stems = stem_text(text, 'porter')
    all_porter_stems.extend(porter_stems)

    snowball_stems = stem_text(text, 'snowball')
    all_snowball_stems.extend(snowball_stems)

# Calculate overall statistics
original_vocab_size = len(set(all_original_words))
porter_vocab_size = len(set(all_porter_stems))
snowball_vocab_size = len(set(all_snowball_stems))

print(f"Dataset vocabulary analysis:")
print(f"Original vocabulary: {original_vocab_size} unique words")
print(f"Porter stemmed: {porter_vocab_size} unique stems ({((original_vocab_size - porter_vocab_size)/original_vocab_size)*100:.1f}% reduction)")
print(f"Snowball stemmed: {snowball_vocab_size} unique stems ({((original_vocab_size - snowball_vocab_size)/original_vocab_size)*100:.1f}% reduction)")

# Show examples of word groups created by stemming
print(f"\n=== WORD GROUPINGS CREATED BY STEMMING ===")

# Create reverse mapping: stem -> original words
porter_groups = defaultdict(set)
for original, stem in zip(all_original_words, all_porter_stems):
    porter_groups[stem].add(original)

# Show interesting groups (stems that group multiple different words)
interesting_groups = {stem: words for stem, words in porter_groups.items()
                     if len(words) > 1 and len(words) < 6}  # Not too many, not too few

print("Examples of words grouped together by Porter stemming:")
for stem, words in list(interesting_groups.items())[:8]:
    word_list = sorted(list(words))
    print(f"  '{stem}' groups: {', '.join(word_list)}")


=== STEMMING: FINDING WORD ROOTS THROUGH ALGORITHMIC RULES ===
Stemming is like pruning a tree - we cut back to the main trunk
It uses simple rules to remove prefixes and suffixes from words

=== COMPREHENSIVE STEMMING DEMONSTRATION ===
Original Word   Porter Stem  Snowball Stem   Notes
----------------------------------------------------------------------

Word family: run
  run           run          run             Same result
  running       run          run             Same result
  runs          run          run             Same result
  ran           ran          ran             Same result
  runner        runner       runner          Same result

Word family: study
  study         studi        studi           Same result
  studying      studi        studi           Same result
  studies       studi        studi           Same result
  studied       studi        studi           Same result
  student       student      student         Same result

Word family: connect
  connect  

In [15]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [20]:
# ===========================
# CELL 7: Lemmatization - Intelligent Word Normalization
# ===========================

print("=== LEMMATIZATION: INTELLIGENT WORD NORMALIZATION ===")
print("Lemmatization is like having a linguistics expert who knows the proper root form of every word")
print("It uses dictionaries and grammatical knowledge to find real word roots\n")

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(word):
    """
    Convert NLTK POS tag to WordNet POS tag format
    WordNet lemmatizer needs specific POS format to work accurately
    """
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {
        "J": wordnet.ADJ,      # Adjective
        "N": wordnet.NOUN,     # Noun
        "V": wordnet.VERB,     # Verb
        "R": wordnet.ADV       # Adverb
    }
    return tag_dict.get(tag, wordnet.NOUN)  # Default to noun if unsure

# Comprehensive test cases to show lemmatization power
lemma_test_cases = [
    # Irregular verbs
    ('went', 'V'), ('came', 'V'), ('saw', 'V'), ('ran', 'V'), ('ate', 'V'),
    # Irregular plurals
    ('children', 'N'), ('mice', 'N'), ('feet', 'N'), ('teeth', 'N'), ('geese', 'N'),
    # Comparative adjectives
    ('better', 'J'), ('worse', 'J'), ('best', 'J'), ('worst', 'J'),
    # Regular forms for comparison
    ('running', 'V'), ('studies', 'N'), ('beautiful', 'J'), ('quickly', 'R'),
    # Challenging cases
    ('was', 'V'), ('were', 'V'), ('am', 'V'), ('is', 'V'), ('are', 'V')
]

print("=== LEMMATIZATION DEMONSTRATION ===")
print("Let's see how lemmatization handles various word forms:")
print(f"{'Original':<12} {'POS':<4} {'Lemma':<12} {'Correct?'}")
print("-" * 45)

for word, pos_hint in lemma_test_cases:
    # Convert our POS hint to WordNet format
    if pos_hint == 'V':
        wordnet_pos = wordnet.VERB
    elif pos_hint == 'N':
        wordnet_pos = wordnet.NOUN
    elif pos_hint == 'J':
        wordnet_pos = wordnet.ADJ
    elif pos_hint == 'R':
        wordnet_pos = wordnet.ADV
    else:
        wordnet_pos = wordnet.NOUN

    lemma = lemmatizer.lemmatize(word, wordnet_pos)

    # Check if this seems like a good lemmatization
    correctness = "✓" if lemma != word else "→"

    print(f"{word:<12} {pos_hint:<4} {lemma:<12} {correctness}")



=== LEMMATIZATION: INTELLIGENT WORD NORMALIZATION ===
Lemmatization is like having a linguistics expert who knows the proper root form of every word
It uses dictionaries and grammatical knowledge to find real word roots

=== LEMMATIZATION DEMONSTRATION ===
Let's see how lemmatization handles various word forms:
Original     POS  Lemma        Correct?
---------------------------------------------
went         V    go           ✓
came         V    come         ✓
saw          V    saw          →
ran          V    run          ✓
ate          V    eat          ✓
children     N    child        ✓
mice         N    mouse        ✓
feet         N    foot         ✓
teeth        N    teeth        →
geese        N    goose        ✓
better       J    good         ✓
worse        J    bad          ✓
best         J    best         →
worst        J    bad          ✓
running      V    run          ✓
studies      N    study        ✓
beautiful    J    beautiful    →
quickly      R    quickly      →
was    